# သင်ခန်းစာ ၀၇ - စီမံကိန်းဖန်တီးရေး ဒီဇိုင်းပုံစံ

ဤမှတ်စုစာအုပ်သည် Microsoft Agent Framework ကို အသုံးပြု၍ AI ကိုယ်စားလှယ်များအတွက် **စီမံကိန်းဖန်တီးရေး ဒီဇိုင်းပုံစံ** ကိုပြသသည်။
သင်သည် ခရီးသွားမော်တော်ယာဉ်စီစဉ်မှုများကို ဖွဲ့စည်းထားသော အသေးစားတာဝန်များအဖြစ် ခွဲထုတ်နည်း၊ အထူးပြု ကိုယ်စားလှယ်များထံ သတ်မှတ်ပေးနည်း၊
နှင့် ရလဒ်အဖြစ်ရရှိသော စီမံကိန်းကို အကောင်အထည်ဖော်နည်း - အားလုံးကို Pydantic မော်ဒယ်များဖြင့် တည်ဆောက်ထားသော ထွက်ရှိချက်များ အသုံးပြုကာ လေ့လာသင်ယူမှာဖြစ်သည်။


## ပြင်ဆင်ခြင်း


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Task Decomposition

Task decomposition is the core of the planning design pattern. Instead of asking a single agent to
handle a complex request end-to-end, we break the problem into smaller, well-defined **subtasks**.
Each subtask is assigned to a specialist agent (e.g., flights, hotels, activities) with clear
priorities and dependency ordering.

This approach provides several benefits:
- **Clarity**: each subtask has a single responsibility.
- **Parallelism**: independent subtasks can run concurrently.
- **Reliability**: failures are isolated to individual subtasks.
- **Budget tracking**: costs are estimated per subtask and rolled up.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## ဖွဲ့စည်းတည်ဆောက်ထားသော ထုတ်လွှင့်မှုဖြင့် စီမံကိန်းကိုယ်စားလှယ် တည်ဆောက်ခြင်း

စီမံကိန်းကိုယ်စားလှယ်သည် **ရှေ့ရုံးညှိနှိုင်းသူ** အဖြစ် လုပ်ဆောင်သည်။ အဆင့်မြင့် ခရီးသွားတောင်းဆိုမှုတစ်ခုကို ပေးလက်ခံပြီး
ဖွဲ့စည်းတည်ဆောက်ထားသော `TravelPlan` တစ်ခု ထုတ်ပေးသည် — တောင်းဆိုမှုအား အသေးစိတ်လုပ်ငန်းငယ်များသို့ ခွဲခြမ်းစိတ်ဖြာပြီး ဦးစားပေးမှုများ သတ်မှတ်ကာ၊
ကောင်စီအရေဖျောက်သို့မဟုတ် အကောင်အထည်ဖော် အလွှာအား တာဝန်ယူဆောင်ရွက်ရန် လိုအပ်သည့် မှီခိုမှုများကို ဖော်ထုတ်ပေးသည်။


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## အထူးပြုကိရိယာများဖြင့် စီမံချက်တစ်ရပ်ကို အပြည့်အဝ လုပ်ဆောင်ခြင်း

ရုံးဖူးဝင်းဝန်ထမ်းသည် စနစ်တကျ စီမံချက်တစ်ရပ်ကို ထုတ်ပေးပြီးနောက်၊ **စောင့်ကြည့်ဝန်ထမ်း**သည် ထိုစီမံချက်ကို အကောင်အထည်ဖော်သည်။
အထူးပြုကိရိယာ တစ်ခုစီသည် အောက်စီးမတစ်ခု (လေယာဉ်ခရီး၊ ဟိုတယ်များ၊ လှုပ်ရှားမှုများ) ကို ကိုင်တွယ်သည်။ စောင့်ကြည့်ဝန်ထမ်းသည်
စီမံချက်ရှိ အောက်စီးမများအား အပြန်အလှန် မူတည်မှုအတိုင်း လှည့်ဖြတ်ပြီး မတူညီသည့် ကိရိယာသို့ တစ်ခုစီ သယ်ယူပို့ဆောင်သည်။
appropriate tool.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Summary

In this lesson you learned the **Planning Design Pattern** for AI agents:

1. **Task Decomposition** — A front desk planning agent breaks a complex travel request into
   structured subtasks using Pydantic models, assigning each to a specialist agent with priorities
   and dependencies.
2. **Structured Output** — By passing a `response_format` the agent returns a validated
   `TravelPlan` object instead of free-form text, making downstream processing reliable.
3. **Plan Execution** — A concierge agent iterates through the subtasks using specialist tools
   (`book_flight`, `reserve_hotel`, `book_activity`) to carry out the plan and report results.

This pattern separates *what to do* (planning) from *how to do it* (execution), making agents
more modular, testable, and easier to extend.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
